# Part 4 – LLM Powered Feature (Track C)
This notebook loads `best_model.pkl`, predicts on three inputs, calls an LLM API, validates JSON, and demonstrates PII guardrails.

In [ ]:
import os,json,re,requests,joblib,pandas as pd
from jsonschema import validate, ValidationError

API_URL="https://openrouter.ai/api/v1/chat/completions"
API_KEY=os.environ.get("LLM_API_KEY")

schema={
"type":"object",
"properties":{
"prediction_label":{"type":"string"},
"confidence_level":{"type":"string"},
"top_reason":{"type":"string"},
"second_reason":{"type":"string"},
"next_step":{"type":"string"}},
"required":["prediction_label","confidence_level","top_reason","second_reason","next_step"]}

def has_pii(text):
    email=r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone=r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email,text) or re.search(phone,text))

def call_llm(system_prompt,user_prompt,temperature=0,max_tokens=512):
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None
    headers={"Authorization":f"Bearer {API_KEY}","Content-Type":"application/json"}
    payload={
      "model":"openai/gpt-4.1-mini",
      "messages":[
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_prompt}],
      "temperature":temperature,
      "max_tokens":max_tokens}
    r=requests.post(API_URL,headers=headers,json=payload)
    if r.status_code!=200:
        print(r.status_code,r.text)
        return None
    return r.json()["choices"][0]["message"]["content"]

print("Guardrail tests:")
print(call_llm("Reply only hello","Email me at test@example.com"))
# Uncomment after setting API key:
# print(call_llm("Reply only with hello","Reply with only the word: hello"))

model=joblib.load("best_model.pkl")

samples=[
{"OrderID":1,"Quantity":3,"Profit":20,"Region_West":1},
{"OrderID":2,"Quantity":8,"Profit":-5,"Region_West":0},
{"OrderID":3,"Quantity":5,"Profit":100,"Region_West":1},
]

system_prompt="""You are an ML explanation assistant.
Output ONLY valid JSON with keys:
prediction_label, confidence_level, top_reason, second_reason, next_step."""

for s in samples:
    df=pd.DataFrame([s])
    pred=model.predict(df)[0]
    prob=model.predict_proba(df)[0].max()
    user=f"Features:{s}\nPrediction:{pred}\nProbability:{prob:.3f}"
    # resp=call_llm(system_prompt,user,temperature=0)
    # Example placeholder:
    resp='{"prediction_label":"Positive","confidence_level":"high","top_reason":"High score","second_reason":"Important features","next_step":"Review"}'
    print(resp)
    try:
        obj=json.loads(resp.strip())
        validate(obj,schema)
        print("Validation: PASS")
    except (json.JSONDecodeError,ValidationError) as e:
        print("Validation: FAIL",e)
